# Task 5: Flee Ability Analysis

This analysis explores the "Flee" ability. We evaluate:
1. Does the average difficulty (level) of available doors affect the decision to flee?
2. How does the number of active skills affect the decision to flee?
3. How do these factors differ between `Control` runs and `Tool` runs?

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact

# Helper to run 2x2 Fisher's Exact Test for each unique value (One vs Rest)
def print_fisher_results(series1, series2, group1_name, group2_name, label):
    print(f"\n--- Fisher's Exact Test: {label} ({group1_name} vs {group2_name}) ---")
    unique_vals = set(series1.dropna().unique()).union(set(series2.dropna().unique()))
    n1, n2 = len(series1), len(series2)
    
    if n1 == 0 or n2 == 0:
        print("Not enough data to run Fisher's Exact Test.")
        return

    results = []
    for val in unique_vals:
        k1 = (series1 == val).sum()
        k2 = (series2 == val).sum()
        
        # 2x2 Contingency Table
        #             Group 1    Group 2
        # Is Val        k1         k2
        # Not Val     n1-k1      n2-k2
        table = [[k1, k2], [n1 - k1, n2 - k2]]
        _, p_val = fisher_exact(table)
        
        results.append({
            'Value': val,
            f'{group1_name}': f"{k1}/{n1} ({k1/n1*100:.1f}%)",
            f'{group2_name}': f"{k2}/{n2} ({k2/n2*100:.1f}%)",
            'p-value': p_val
        })
    
    df_res = pd.DataFrame(results).sort_values('p-value')
    print(df_res.to_string(index=False))
    print("-" * 60)

# Load Flee data
with open('../scripts/parsed_task5.json', 'r') as f:
    flee_data = json.load(f)
df_flee = pd.DataFrame(flee_data)

if df_flee.empty:
    print("No flee actions found in the current datasets.")
else:
    print(f"Loaded {len(df_flee)} Flee events.")
    
    # Split Control and Tool for Flee comparisons
    df_flee_control = df_flee[df_flee['mode'] == 'control']
    df_flee_tool = df_flee[df_flee['mode'] == 'tool']

    # Fisher Test: Control vs Skill (Active Skills Count)
    # Note: avg_door_level is continuous, so Fisher's Exact Test is only applied to the discrete active_skills_count
    print_fisher_results(df_flee_control['active_skills_count'], df_flee_tool['active_skills_count'], 'Control', 'Tool', 'Active Skills Count (Flee)')

    # ==========================================================
    # 1. Compare Control vs Skill: Average Door Level when Fleeing
    # ==========================================================
    plt.figure(figsize=(10, 6))
    ax1 = sns.boxplot(data=df_flee, x='mode', y='avg_door_level', palette='Set1')
    sns.stripplot(data=df_flee, x='mode', y='avg_door_level', color='black', alpha=0.5, jitter=True, ax=ax1)
    
    # Add median values as text on the boxplot
    medians = df_flee.groupby(['mode'])['avg_door_level'].median()
    # Get the x-axis labels to match the positions
    xticklabels = [t.get_text() for t in ax1.get_xticklabels()]
    for i, mode in enumerate(xticklabels):
        if mode in medians:
            median_val = medians[mode]
            ax1.text(i, median_val, f'Median: {median_val:.2f}', 
                     horizontalalignment='center', verticalalignment='bottom',
                     size='medium', color='white', weight='bold',
                     bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', boxstyle='round,pad=0.2'))

    plt.title('Average Level of Available Doors When Fleeing (Control vs Skill)')
    plt.xlabel('Mode')
    plt.ylabel('Average Door Level')
    plt.tight_layout()
    plt.show()

    # ==========================================================
    # 2. Active Skills Count During Flee (Control vs Skill)
    # ==========================================================
    plt.figure(figsize=(10, 6))
    ax2 = sns.countplot(data=df_flee, x='active_skills_count', hue='mode', palette='Set3')
    plt.title('Number of Active Skills When Fleeing')
    plt.xlabel('Active Skills Count')
    plt.ylabel('Count')
    
    # Add value labels to the bars
    for container in ax2.containers:
        ax2.bar_label(container, padding=3)
        
    plt.tight_layout()
    plt.show()

No flee actions found in the current datasets.
